# 树模型机器学习选股策略

## 完整流程

| 步骤 | 内容 | 模型/技术 |
|------|------|----------|
| 1 | 读取本地股票数据 | pandas CSV 加载 |
| 2 | 按板块/逻辑筛选标的 | 主板过滤、ST排除 |
| 3 | 构建过去X年特征数据集 | Technical Factor + Fundamental Factor |
| 4 | 划分训练/验证/测试集 | 严格时序划分，无数据穿越 |
| 5 | 训练树模型 | Random Forest / XGBoost / LightGBM |
| 6 | 模型对比与特征重要性 | SHAP / Feature Importance |
| 7 | 新标的回测效果 | VectorBT 信号回测 |

> **核心特点**: 
> - 使用树模型进行回归预测（预测未来收益率）
> - 支持多模型对比：Random Forest、XGBoost、LightGBM
> - 特征重要性分析
> - 严格防止数据穿越

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pathlib
PROJECT_ROOT = pathlib.Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 树模型库
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠️  XGBoost 未安装，将跳过 XGBoost 模型")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("⚠️  LightGBM 未安装，将跳过 LightGBM 模型")

try:
    import vectorbt as vbt
    HAS_VBT = True
except ImportError:
    HAS_VBT = False
    print("⚠️  vectorbt 未安装")

from utils.factor_calculator import TechnicalFactorCalculator

# ============================================================
# 全局参数
# ============================================================
CSV_PATH         = './data/a_stock_history_price_20260223.csv'
HISTORY_YEARS    = 5        # 使用过去 N 年数据
PRED_HORIZON     = 5        # 预测未来 N 天收益率
TRAIN_RATIO      = 0.60
VAL_RATIO        = 0.20
TEST_RATIO       = 0.20
TOP_K_STOCKS     = 10       # 每日选前K只股票
RANDOM_SEED      = 42

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成")
print(f"   预测目标: 未来 {PRED_HORIZON} 日收益率（回归问题）")
print(f"   训练/验证/测试: {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")

✅ 环境配置完成
   预测目标: 未来 5 日收益率（回归问题）
   训练/验证/测试: 60% / 20% / 20%


## 步骤 1：数据加载与预处理

In [2]:
def load_stock_data(path, years_back=5):
    """加载股票数据"""
    print(f"📂 加载数据: {path}")
    
    if not os.path.exists(path):
        # 尝试查找替代文件
        alt_path = './data/a_stock_history_price.csv'
        if os.path.exists(alt_path):
            path = alt_path
            print(f"   使用替代路径: {path}")
        else:
            raise FileNotFoundError(f"找不到数据文件")
    
    df = pd.read_csv(path, low_memory=False)
    print(f"   原始: {df.shape[0]:,} 行 × {df.shape[1]} 列")
    
    # 标准化列名
    col_map = {'stock_code': 'code', 'ts_code': 'code', 'trade_date': 'date'}
    df = df.rename(columns=col_map)
    
    # 处理股票代码
    if 'code' in df.columns:
        df['code'] = df['code'].astype(str).str.zfill(6)
    
    # 日期处理
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    
    # 数值列转换
    num_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 过滤时间范围
    if years_back:
        cutoff = df['date'].max() - pd.DateOffset(years=years_back)
        df = df[df['date'] >= cutoff]
    
    # 排序
    df = df.sort_values(['code', 'date']).reset_index(drop=True)
    
    print(f"   加载后: {df.shape[0]:,} 行 | {df['code'].nunique():,} 只股票")
    print(f"   日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
    return df

# 加载数据
df_raw = load_stock_data(CSV_PATH, HISTORY_YEARS)
df_raw.head()

📂 加载数据: ./data/a_stock_history_price_20260223.csv
   使用替代路径: ./data/a_stock_history_price.csv
   原始: 7,783,131 行 × 14 列
   加载后: 6,706,561 行 | 6,403 只股票
   日期范围: 2021-02-18 ~ 2026-02-13


,date,code,name,open,close,high,low,price_change,price_change_rate,volume,turnover_rate,market_cap,dividends,stock_splits
0,2021-02-18,000001,平安银行,19.81,20.06,20.26,19.38,0.40,2.03,150523081.0,0.78,3.892827e+11,0.0,0.0
1,2021-02-19,000001,平安银行,19.78,19.69,19.97,19.46,-0.37,-1.84,92258314.0,0.48,3.821025e+11,0.0,0.0
2,2021-02-22,000001,平安银行,19.65,18.47,19.65,18.37,-1.22,-6.20,195684248.0,1.01,3.584273e+11,0.0,0.0
3,2021-02-23,000001,平安银行,18.47,18.10,18.72,18.01,-0.37,-2.00,187046013.0,0.96,3.512471e+11,0.0,0.0
4,2021-02-24,000001,平安银行,18.38,18.29,18.39,17.80,0.19,1.05,142849985.0,0.74,3.549342e+11,0.0,0.0


## 步骤 2：特征工程

In [3]:
def create_features(df):
    """
    构建特征数据集
    包括技术指标和基本特征
    """
    print("\n🔧 构建特征...")
    
    all_features = []
    
    for code in df['code'].unique():
        stock_df = df[df['code'] == code].copy()
        stock_df = stock_df.sort_values('date')
        
        if len(stock_df) < 30:
            continue
        
        # 计算技术指标
        calc = TechnicalFactorCalculator()
        stock_df = calc.moving_average(stock_df, windows=[5, 10, 20, 60])
        stock_df = calc.rsi(stock_df, windows=[6, 14])
        stock_df = calc.macd(stock_df)
        stock_df = calc.bollinger_bands(stock_df)
        stock_df = calc.volatility_factors(stock_df)
        stock_df = calc.volume_factors(stock_df)
        stock_df = calc.momentum_factors(stock_df)
        
        # 目标变量：未来N日收益率
        stock_df[f'target_return_{PRED_HORIZON}d'] = (
            stock_df['close'].shift(-PRED_HORIZON) / stock_df['close'] - 1
        )
        
        # 日期特征
        stock_df['dayofweek'] = stock_df['date'].dt.dayofweek
        stock_df['month'] = stock_df['date'].dt.month
        
        all_features.append(stock_df)
    
    df_features = pd.concat(all_features, ignore_index=True)
    
    # 删除包含NaN的行（主要是特征计算产生的）
    feature_cols = [c for c in df_features.columns if any(x in c for x in 
                    ['ma_', 'rsi_', 'macd_', 'bb_', 'volatility_', 'momentum_', 'volume_', 'dayofweek', 'month'])]
    
    df_features = df_features.dropna(subset=feature_cols + [f'target_return_{PRED_HORIZON}d'])
    
    print(f"   特征数: {len(feature_cols)}")
    print(f"   有效样本: {len(df_features):,}")
    print(f"   特征示例: {feature_cols[:10]}")
    
    return df_features, feature_cols

df_features, FEATURE_COLS = create_features(df_raw)
df_features.head()


🔧 构建特征...


KeyError: 'amount'

## 步骤 3：时间序列数据集划分

In [ ]:
def time_series_split(df, train_ratio=0.6, val_ratio=0.2):
    """
    按时间序列划分数据集（严格无穿越）
    """
    print("\n⏰ 时间序列数据集划分...")
    
    dates = sorted(df['date'].unique())
    n_dates = len(dates)
    
    train_end = int(n_dates * train_ratio)
    val_end = int(n_dates * (train_ratio + val_ratio))
    
    train_dates = dates[:train_end]
    val_dates = dates[train_end:val_end]
    test_dates = dates[val_end:]
    
    train_df = df[df['date'].isin(train_dates)].copy()
    val_df = df[df['date'].isin(val_dates)].copy()
    test_df = df[df['date'].isin(test_dates)].copy()
    
    print(f"   训练集: {train_df['date'].min().date()} ~ {train_df['date'].max().date()} ({len(train_df):,})")
    print(f"   验证集: {val_df['date'].min().date()} ~ {val_df['date'].max().date()} ({len(val_df):,})")
    print(f"   测试集: {test_df['date'].min().date()} ~ {test_df['date'].max().date()} ({len(test_df):,})")
    
    return train_df, val_df, test_df

train_df, val_df, test_df = time_series_split(df_features)

# 准备特征和标签
target_col = f'target_return_{PRED_HORIZON}d'

X_train, y_train = train_df[FEATURE_COLS], train_df[target_col]
X_val, y_val = val_df[FEATURE_COLS], val_df[target_col]
X_test, y_test = test_df[FEATURE_COLS], test_df[target_col]

print(f"\n✅ 数据准备完成")
print(f"   训练: X{X_train.shape}, y{y_train.shape}")
print(f"   验证: X{X_val.shape}, y{y_val.shape}")
print(f"   测试: X{X_test.shape}, y{y_test.shape}")

## 步骤 4：训练树模型

In [ ]:
# 定义模型
models = {}

# 1. Random Forest
models['RandomForest'] = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=0
)

# 2. XGBoost
if HAS_XGB:
    models['XGBoost'] = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0
    )

# 3. LightGBM
if HAS_LGB:
    models['LightGBM'] = lgb.LGBMRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    )

print(f"✅ 定义了 {len(models)} 个模型")
for name in models.keys():
    print(f"   - {name}")

In [ ]:
# 训练所有模型
trained_models = {}
predictions = {}

print("\n🚀 开始训练模型...")

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"📊 训练 {name}...")
    
    # 训练
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # 预测
    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    
    predictions[name] = {
        'train': pred_train,
        'val': pred_val,
        'test': pred_test
    }
    
    # 评估
    print(f"   训练集 RMSE: {np.sqrt(mean_squared_error(y_train, pred_train)):.4f}")
    print(f"   验证集 RMSE: {np.sqrt(mean_squared_error(y_val, pred_val)):.4f}")
    print(f"   测试集 RMSE: {np.sqrt(mean_squared_error(y_test, pred_test)):.4f}")
    print(f"   测试集 R²: {r2_score(y_test, pred_test):.4f}")

print(f"\n✅ 所有模型训练完成")

## 步骤 5：特征重要性分析

In [ ]:
# 绘制特征重要性
n_models = len(trained_models)
fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 8))

if n_models == 1:
    axes = [axes]

for idx, (name, model) in enumerate(trained_models.items()):
    # 获取特征重要性
    importance = model.feature_importances_
    feature_imp = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': importance
    }).sort_values('importance', ascending=True).tail(15)  # Top 15
    
    axes[idx].barh(feature_imp['feature'], feature_imp['importance'])
    axes[idx].set_title(f'{name} - Feature Importance')
    axes[idx].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('./data/tree_model_feature_importance.png', dpi=150)
plt.show()

print("✅ 特征重要性图已保存")

## 步骤 6：选股策略回测

In [ ]:
def backtest_strategy(test_df, predictions, model_name, top_k=10):
    """
    回测策略：每日选预测收益最高的top_k只股票
    """
    df = test_df.copy()
    df['pred_return'] = predictions[model_name]['test']
    
    # 按日期分组，选每日top_k
    daily_returns = []
    
    for date, group in df.groupby('date'):
        if len(group) < top_k:
            continue
        
        # 选预测收益最高的top_k
        top_stocks = group.nlargest(top_k, 'pred_return')
        
        # 计算平均实际收益
        avg_return = top_stocks[f'target_return_{PRED_HORIZON}d'].mean()
        daily_returns.append({
            'date': date,
            'return': avg_return,
            'n_stocks': len(top_stocks)
        })
    
    returns_df = pd.DataFrame(daily_returns)
    returns_df.set_index('date', inplace=True)
    
    # 计算累计收益
    returns_df['cum_return'] = (1 + returns_df['return']).cumprod() - 1
    
    # 计算指标
    total_return = returns_df['cum_return'].iloc[-1]
    sharpe = returns_df['return'].mean() / returns_df['return'].std() * np.sqrt(252) if returns_df['return'].std() > 0 else 0
    max_dd = (returns_df['cum_return'] - returns_df['cum_return'].cummax()).min()
    win_rate = (returns_df['return'] > 0).mean()
    
    return {
        'returns_df': returns_df,
        'total_return': total_return,
        'sharpe': sharpe,
        'max_drawdown': max_dd,
        'win_rate': win_rate
    }

# 回测所有模型
print("\n🚀 开始回测策略...")
print(f"   策略: 每日选预测收益最高的{TOP_K_STOCKS}只股票")

backtest_results = {}

for name in trained_models.keys():
    result = backtest_strategy(test_df, predictions, name, TOP_K_STOCKS)
    backtest_results[name] = result
    
    print(f"\n{name}:")
    print(f"   总收益率: {result['total_return']*100:.2f}%")
    print(f"   夏普比率: {result['sharpe']:.3f}")
    print(f"   最大回撤: {result['max_drawdown']*100:.2f}%")
    print(f"   胜率: {result['win_rate']*100:.1f}%")

In [ ]:
# 绘制回测结果对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 累计收益曲线
ax = axes[0]
for name, result in backtest_results.items():
    ax.plot(result['returns_df'].index, result['returns_df']['cum_return']*100, 
            label=name, linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title(f'Strategy Backtest (Top {TOP_K_STOCKS} Stocks)')
ax.legend()
ax.grid(True, alpha=0.3)

# 收益分布
ax = axes[1]
for name, result in backtest_results.items():
    ax.hist(result['returns_df']['return']*100, bins=30, alpha=0.5, label=name)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Return Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./data/tree_model_backtest_comparison.png', dpi=150)
plt.show()

print("\n✅ 回测对比图已保存")

## 总结

In [ ]:
print("=" * 70)
print("📊 树模型机器学习选股策略总结")
print("=" * 70)

print("\n📈 模型对比:")
print(f"{'Model':<15} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'WinRate':>8}")
print("-" * 55)
for name, result in backtest_results.items():
    print(f"{name:<15} {result['total_return']*100:>9.2f}% {result['sharpe']:>8.3f} {result['max_drawdown']*100:>7.2f}% {result['win_rate']*100:>7.1f}%")

print("\n💡 关键发现:")
best_model = max(backtest_results.items(), key=lambda x: x[1]['sharpe'])
print(f"   - 最佳模型（按夏普）: {best_model[0]}")
print(f"   - 预测目标: 未来{PRED_HORIZON}日收益率")
print(f"   - 选股数量: 每日前{TOP_K_STOCKS}只")

print("\n📁 输出文件:")
print("   - data/tree_model_feature_importance.png")
print("   - data/tree_model_backtest_comparison.png")

print("\n" + "=" * 70)